# YOLO11m — Evaluation Only

**재학습 없음.** 이미 완료된 `baseline / tune_a / tune_b` 체크포인트 3개를 V6.0.5의 **고정 upstream Validation 2,433장**으로만 평가한다.

- Primary metric: `mAP@[0.75:0.95]`
- Train/Val 재분할 금지 (`preserve_exact_upstream`)
- Independent Test 생성 금지
- `model.train()` 호출 없음
- `annotations.json` bridge 불필요: 기존 YOLO TXT 라벨을 직접 ground truth로 사용
- 대형 Validation에서는 V6.0.5 원본과 같은 coordinate policy search 사용
> **V3 fix:** Validation 원본 경로를 `images/val`, `labels/val`로 수정했습니다.

> **V4 fix:** Drive I/O 끊김 자동 재마운트/재시도, 이미 로컬에 복사된 Validation 이미지는 재사용, 정상 평가+Drive 저장 완료 후 Colab 런타임 자동 해제.


## 0. 패키지

In [7]:
import subprocess, sys
PINNED = [
    "ultralytics==8.4.116",
    "torchmetrics==1.9.0",
    "pycocotools==2.0.11",
    "tqdm==4.67.1",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED])
print("Packages ready.")

Packages ready.


## 1. 환경 및 경로

In [8]:
# 이 셀부터 실행해도 필요한 패키지가 없으면 자동 설치한다.
import importlib.util, subprocess, sys
_REQUIRED = {
    "ultralytics": "ultralytics==8.4.116",
    "torchmetrics": "torchmetrics==1.9.0",
    "pycocotools": "pycocotools==2.0.11",
    "tqdm": "tqdm==4.67.1",
}
_missing = [pkg for mod, pkg in _REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("Installing missing packages:", _missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

from pathlib import Path
from datetime import datetime
import gc, json, math, os, shutil, time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from ultralytics import YOLO
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from google.colab import drive
from IPython.display import display

drive.mount("/content/drive")

TARGET_MODEL = "YOLO11m"
MODEL_PREFIX = "yolo11m"
IMAGE_SIZE = 960
EXPECTED_CLASSES = 118
EXPECTED_TRAIN_IMAGES = 9763
EXPECTED_VAL_IMAGES = 2433

COMPETITION_METRIC = "mAP@[0.75:0.95]"
COMPETITION_IOU_THRESHOLDS = [0.75, 0.80, 0.85, 0.90, 0.95]
CONFIDENCE_CANDIDATES = [0.001, 0.01, 0.03, 0.05, 0.10, 0.20, 0.30, 0.50]
RAW_PREDICTION_CONFIDENCE = 0.001
NMS_IOU_THRESHOLD = 0.70
MAX_DETECTIONS = 300
COCO_EVAL_MAX_DETECTIONS = 100
INFERENCE_BATCH = 4

WEEK2_ROOT = Path("/content/drive/MyDrive/baby_kangaroo/week2")
PREPROCESS_ROOT = WEEK2_ROOT / "데이터전처리" / "yolo_전처리" / "pill_yolo_full_v6_0_preprocessed"
PIPELINE_ROOT = WEEK2_ROOT / "공통파이프라인"
CHECKPOINT_DIR = PIPELINE_ROOT / "checkpoints"
MANIFEST_DIR = PIPELINE_ROOT / "manifests"
REPORT_DIR = PIPELINE_ROOT / "reports"

LOCAL_ROOT = Path(f"/content/baby_kangaroo_eval_only_{MODEL_PREFIX}")
LOCAL_VAL_DIR = LOCAL_ROOT / "val_images"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for p in [PREPROCESS_ROOT, CHECKPOINT_DIR, MANIFEST_DIR, REPORT_DIR]:
    if not p.exists():
        raise FileNotFoundError(f"Required path not found: {p}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required for this evaluator.")

print("Target:", TARGET_MODEL)
print("Mode: EVALUATION ONLY (model.train is never called)")
print("Preprocess:", PREPROCESS_ROOT)
print("Checkpoints:", CHECKPOINT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Target: YOLO11m
Mode: EVALUATION ONLY (model.train is never called)
Preprocess: /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed
Checkpoints: /content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/checkpoints


## 2. 기존 3개 체크포인트 + 고정 split 검증

In [9]:
STAGES = ("baseline", "tune_a", "tune_b")
checkpoint_info = {}

for stage in STAGES:
    matches = sorted(CHECKPOINT_DIR.glob(f"{MODEL_PREFIX}_{stage}_*_best.pt"))
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one checkpoint for {MODEL_PREFIX}_{stage}, found {len(matches)}: "
            f"{[p.name for p in matches]}"
        )
    ckpt = matches[0]
    stem = ckpt.name[:-len("_best.pt")]
    manifest_path = MANIFEST_DIR / f"{stem}.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Manifest missing: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("training_completed") is not True:
        raise RuntimeError(f"Checkpoint is not marked completed: {ckpt.name}")
    if manifest.get("model_name") != TARGET_MODEL:
        raise RuntimeError(
            f"Model mismatch for {ckpt.name}: manifest={manifest.get('model_name')} target={TARGET_MODEL}"
        )
    checkpoint_info[stage] = {
        "path": ckpt,
        "stem": stem,
        "manifest": manifest,
        "manifest_path": manifest_path,
    }

split_fingerprints = {info["manifest"].get("split_fingerprint") for info in checkpoint_info.values()}
if len(split_fingerprints) != 1 or None in split_fingerprints:
    raise RuntimeError(f"Three checkpoints do not share one split fingerprint: {split_fingerprints}")
SPLIT_FINGERPRINT = next(iter(split_fingerprints))

split_path = MANIFEST_DIR / f"preserved_split_{SPLIT_FINGERPRINT[:16]}.json"
if not split_path.is_file():
    raise FileNotFoundError(f"Fixed upstream split file missing: {split_path}")
split_contract = json.loads(split_path.read_text(encoding="utf-8"))

if split_contract.get("split_policy") != "preserve_exact_upstream":
    raise RuntimeError(f"Unexpected split policy: {split_contract.get('split_policy')}")
if split_contract.get("independent_test_available") is not False:
    raise RuntimeError("This evaluator expects the V6.0.5 Train/Val-only contract.")

train_files = list(split_contract["train_files"])
val_files = list(split_contract["val_files"])
if len(train_files) != EXPECTED_TRAIN_IMAGES or len(val_files) != EXPECTED_VAL_IMAGES:
    raise RuntimeError(
        f"Unexpected fixed split size: train={len(train_files)}, val={len(val_files)}"
    )
if set(train_files) & set(val_files):
    raise RuntimeError("Train/Val file leakage detected in preserved split.")

print(f"Fixed split verified: Train {len(train_files):,} / Val {len(val_files):,} / Test 없음")
print("Split fingerprint:", SPLIT_FINGERPRINT)
for stage, info in checkpoint_info.items():
    print(f"{stage:8s} -> {info['path'].name}")

Fixed split verified: Train 9,763 / Val 2,433 / Test 없음
Split fingerprint: cc5d16d3fe042c5297aaad5c8f2fe469ebfecfb1268223f9a4f3465bcc823154
baseline -> yolo11m_baseline_21cc7ae361705449_best.pt
tune_a   -> yolo11m_tune_a_4abb3f0ea436d561_best.pt
tune_b   -> yolo11m_tune_b_c35a806678e468d7_best.pt


## 3. 고정 Validation 2,433장만 로컬에 준비

In [10]:
mapping_path = PREPROCESS_ROOT / "class_mapping.csv"
manifest_csv_path = PREPROCESS_ROOT / "dataset_manifest.csv"

# V4: V6.0.5 fixed Validation lives under images/val and labels/val.
image_root = PREPROCESS_ROOT / "images" / "val"
label_root = PREPROCESS_ROOT / "labels" / "val"

# ---------------------------------------------------------------------------
# Drive I/O 안정화
# - Errno 107 등 Drive FUSE 연결 오류가 나면 force_remount 후 재시도
# - 이미 /content 에 정상 복사된 이미지는 다시 복사하지 않음
# ---------------------------------------------------------------------------
DRIVE_IO_RETRIES = 4
DRIVE_RETRY_SLEEP = 3.0

def _force_remount_drive():
    print("Google Drive 연결을 다시 설정합니다...")
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(1.0)
    drive.mount("/content/drive", force_remount=True)
    time.sleep(2.0)

def _drive_retry(operation_name, fn, retries=DRIVE_IO_RETRIES):
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            return fn()
        except (OSError, IOError) as exc:
            last_exc = exc
            print(
                f"[Drive I/O retry {attempt}/{retries}] "
                f"{operation_name}: {type(exc).__name__}: {exc}"
            )
            if attempt >= retries:
                break
            _force_remount_drive()
            time.sleep(DRIVE_RETRY_SLEEP)
    raise RuntimeError(
        f"Drive I/O failed after {retries} attempts: {operation_name}"
    ) from last_exc

def _require_drive_path(path):
    ok = _drive_retry(f"check {path}", lambda: path.exists())
    if not ok:
        raise FileNotFoundError(f"Preprocessing resource missing: {path}")

for p in [mapping_path, manifest_csv_path, image_root, label_root]:
    _require_drive_path(p)

mapping_df = _drive_retry("read class_mapping.csv", lambda: pd.read_csv(mapping_path))
required_mapping = {"yolo_class_id", "original_category_id"}
if not required_mapping.issubset(mapping_df.columns):
    raise RuntimeError(
        f"class_mapping.csv missing columns: "
        f"{required_mapping - set(mapping_df.columns)}"
    )
mapping_df["yolo_class_id"] = pd.to_numeric(
    mapping_df["yolo_class_id"], errors="raise"
).astype(int)
if sorted(mapping_df["yolo_class_id"].tolist()) != list(range(EXPECTED_CLASSES)):
    raise RuntimeError("YOLO class IDs are not continuous 0..117.")

dataset_manifest_df = _drive_retry(
    "read dataset_manifest.csv",
    lambda: pd.read_csv(manifest_csv_path),
)
if "file_name" not in dataset_manifest_df.columns:
    raise RuntimeError("dataset_manifest.csv has no file_name column.")

dataset_files = set(dataset_manifest_df["file_name"].astype(str))
missing_val = sorted(set(val_files) - dataset_files)
if missing_val:
    raise RuntimeError(
        f"Preserved Validation contains files absent from dataset manifest: "
        f"{missing_val[:5]}"
    )

if "object_count" in dataset_manifest_df.columns:
    max_objects_per_image = int(
        pd.to_numeric(
            dataset_manifest_df["object_count"], errors="coerce"
        ).fillna(0).max()
    )
else:
    max_objects_per_image = 4

max_objects_per_image = max(1, max_objects_per_image)
TOP_K_CANDIDATES = sorted({
    max_objects_per_image,
    max_objects_per_image * 2,
    max_objects_per_image * 3,
}) + [None]
print("Top-K candidates:", TOP_K_CANDIDATES)

# V4: 기존 LOCAL_VAL_DIR를 지우지 않는다.
# 같은 Colab 런타임에서 staging을 재실행하면 이미 복사된 정상 파일은 재사용한다.
LOCAL_VAL_DIR.mkdir(parents=True, exist_ok=True)

ground_truth_by_file = {}

def read_yolo_gt_text(text, label_path):
    boxes, labels = [], []
    text = text.strip()
    if not text:
        return (
            torch.empty((0, 4), dtype=torch.float32),
            torch.empty((0,), dtype=torch.int64),
        )

    for line_no, line in enumerate(text.splitlines(), 1):
        parts = line.split()
        if len(parts) != 5:
            raise RuntimeError(f"Invalid YOLO label: {label_path}:{line_no}")

        cls, cx, cy, w, h = map(float, parts)
        cls_i = int(cls)
        if cls != cls_i or not (0 <= cls_i < EXPECTED_CLASSES):
            raise RuntimeError(f"Invalid class ID: {label_path}:{line_no}")

        raw_x1 = (cx - w / 2.0) * IMAGE_SIZE
        raw_y1 = (cy - h / 2.0) * IMAGE_SIZE
        raw_x2 = (cx + w / 2.0) * IMAGE_SIZE
        raw_y2 = (cy + h / 2.0) * IMAGE_SIZE

        # YOLO txt 소수점 저장에 따른 미세한 경계 오차만 허용한다.
        boundary_tol_px = IMAGE_SIZE * 1e-5  # 960 -> 0.0096 px
        if (
            raw_x1 < -boundary_tol_px
            or raw_y1 < -boundary_tol_px
            or raw_x2 > IMAGE_SIZE + boundary_tol_px
            or raw_y2 > IMAGE_SIZE + boundary_tol_px
        ):
            raise RuntimeError(
                f"BBox exceeds image by more than rounding tolerance at "
                f"{label_path}:{line_no}: "
                f"{(raw_x1, raw_y1, raw_x2, raw_y2)}"
            )

        x1 = min(max(raw_x1, 0.0), float(IMAGE_SIZE))
        y1 = min(max(raw_y1, 0.0), float(IMAGE_SIZE))
        x2 = min(max(raw_x2, 0.0), float(IMAGE_SIZE))
        y2 = min(max(raw_y2, 0.0), float(IMAGE_SIZE))

        if x2 <= x1 or y2 <= y1:
            raise RuntimeError(
                f"Empty bbox after clipping at {label_path}:{line_no}: "
                f"{(x1, y1, x2, y2)}"
            )

        boxes.append([x1, y1, x2, y2])
        labels.append(cls_i)

    return (
        torch.tensor(boxes, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.int64),
    )

def _copy_validation_image(src, dst):
    # 이전 시도에서 0-byte/부분 파일이 남았으면 다시 복사한다.
    def _copy_once():
        src_size = src.stat().st_size
        if dst.exists() and dst.stat().st_size == src_size and src_size > 0:
            return "reused"
        tmp = dst.with_suffix(dst.suffix + ".part")
        if tmp.exists():
            tmp.unlink()
        shutil.copyfile(src, tmp)
        if tmp.stat().st_size != src_size:
            raise OSError(
                f"Copied size mismatch: src={src_size}, dst={tmp.stat().st_size}"
            )
        os.replace(tmp, dst)
        return "copied"

    return _drive_retry(f"copy {src.name}", _copy_once)

reused_count = 0
copied_count = 0

for file_name in tqdm(
    val_files,
    desc="Staging fixed Validation",
    unit="image",
):
    src = image_root / file_name
    lbl = label_root / f"{Path(file_name).stem}.txt"
    dst = LOCAL_VAL_DIR / file_name

    src_exists = _drive_retry(
        f"check image {file_name}",
        lambda p=src: p.is_file(),
    )
    if not src_exists:
        raise FileNotFoundError(f"Validation image missing: {src}")

    lbl_exists = _drive_retry(
        f"check label {lbl.name}",
        lambda p=lbl: p.is_file(),
    )
    if not lbl_exists:
        raise FileNotFoundError(f"Validation label missing: {lbl}")

    status = _copy_validation_image(src, dst)
    if status == "reused":
        reused_count += 1
    else:
        copied_count += 1

    label_text = _drive_retry(
        f"read label {lbl.name}",
        lambda p=lbl: p.read_text(encoding="utf-8"),
    )
    gt_boxes, gt_labels = read_yolo_gt_text(label_text, lbl)
    ground_truth_by_file[file_name] = {
        "boxes": gt_boxes,
        "labels": gt_labels,
    }

if len(ground_truth_by_file) != EXPECTED_VAL_IMAGES:
    raise RuntimeError("Validation ground-truth count mismatch.")

local_files = [p for p in LOCAL_VAL_DIR.iterdir() if p.is_file() and not p.name.endswith(".part")]
if len(local_files) != EXPECTED_VAL_IMAGES:
    raise RuntimeError(
        f"Local Validation image count mismatch: {len(local_files)} "
        f"!= {EXPECTED_VAL_IMAGES}"
    )

print(
    f"Local fixed Validation ready: {len(ground_truth_by_file):,} images "
    f"(reused={reused_count:,}, copied={copied_count:,})"
)


Top-K candidates: [6, 12, 18, None]


Staging fixed Validation:   0%|          | 0/2433 [00:00<?, ?image/s]

Local fixed Validation ready: 2,433 images (reused=2,433, copied=0)


## 4. Competition metric / policy search

In [11]:
TARGETS_TM = [
    {
        "boxes": ground_truth_by_file[file_name]["boxes"],
        "labels": ground_truth_by_file[file_name]["labels"],
    }
    for file_name in val_files
]

def filter_one_prediction(pred, threshold, top_k):
    scores = pred["scores"]
    keep = torch.where(scores >= float(threshold))[0]
    if keep.numel():
        keep = keep[torch.argsort(scores[keep], descending=True)]
    if top_k is not None:
        keep = keep[:int(top_k)]
    return {
        "boxes": pred["boxes"][keep],
        "labels": pred["labels"][keep],
        "scores": pred["scores"][keep],
    }

def competition_map(predictions_by_file, threshold, top_k):
    preds_tm = [
        filter_one_prediction(predictions_by_file[file_name], threshold, top_k)
        for file_name in val_files
    ]
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=COMPETITION_IOU_THRESHOLDS,
        max_detection_thresholds=[1, 10, COCO_EVAL_MAX_DETECTIONS],
        class_metrics=False,
    )
    metric.update(preds_tm, TARGETS_TM)
    result = metric.compute()
    value = float(result["map"].cpu().item())
    del metric, preds_tm, result
    gc.collect()
    return value

def select_policy(predictions_by_file):
    # V6.0.5 원본과 동일한 대형 Validation용 coordinate search:
    # 32개 full grid 대신 8개 confidence + 선택 confidence에서 Top-K만 평가.
    rows = []
    best_threshold = None
    best_score = -1.0

    for threshold in CONFIDENCE_CANDIDATES:
        score = competition_map(predictions_by_file, threshold, None)
        rows.append({
            "threshold": float(threshold),
            "top_k": None,
            "top_k_label": "unlimited",
            "competition_mAP": score,
            "search_mode": "coordinate",
        })
        if score > best_score:
            best_threshold, best_score = float(threshold), score

    for top_k in TOP_K_CANDIDATES:
        if top_k is None:
            continue
        score = competition_map(predictions_by_file, best_threshold, int(top_k))
        rows.append({
            "threshold": best_threshold,
            "top_k": int(top_k),
            "top_k_label": str(int(top_k)),
            "competition_mAP": score,
            "search_mode": "coordinate",
        })

    table = pd.DataFrame(rows)
    table["top_k_sort"] = table["top_k"].apply(
        lambda x: 10**9 if pd.isna(x) else int(x)
    )
    best = table.sort_values(
        ["competition_mAP", "threshold", "top_k_sort"],
        ascending=[False, True, True],
    ).iloc[0]
    selected_top_k = None if pd.isna(best["top_k"]) else int(best["top_k"])
    return (
        float(best["threshold"]),
        selected_top_k,
        float(best["competition_mAP"]),
        table.drop(columns=["top_k_sort"]),
    )

## 5. 체크포인트 추론

In [12]:
def infer_checkpoint(checkpoint_path):
    model = YOLO(str(checkpoint_path))

    # CUDA 초기화 시간은 본 추론 시간에서 제외한다.
    warmup_file = LOCAL_VAL_DIR / val_files[0]
    model.predict(
        source=str(warmup_file),
        conf=RAW_PREDICTION_CONFIDENCE,
        iou=NMS_IOU_THRESHOLD,
        max_det=MAX_DETECTIONS,
        imgsz=IMAGE_SIZE,
        device=0,
        verbose=False,
    )
    torch.cuda.synchronize()

    started = time.perf_counter()
    predictions = {}
    stream = model.predict(
        source=str(LOCAL_VAL_DIR),
        conf=RAW_PREDICTION_CONFIDENCE,
        iou=NMS_IOU_THRESHOLD,
        max_det=MAX_DETECTIONS,
        imgsz=IMAGE_SIZE,
        device=0,
        batch=INFERENCE_BATCH,
        verbose=False,
        stream=True,
    )
    for result in tqdm(stream, total=len(val_files), desc=f"{TARGET_MODEL} inference", unit="image"):
        file_name = Path(result.path).name
        predictions[file_name] = {
            "boxes": result.boxes.xyxy.detach().cpu().to(torch.float32).reshape(-1, 4),
            "labels": result.boxes.cls.detach().cpu().to(torch.int64).reshape(-1),
            "scores": result.boxes.conf.detach().cpu().to(torch.float32).reshape(-1),
        }
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    if set(predictions) != set(val_files):
        missing = sorted(set(val_files) - set(predictions))
        extra = sorted(set(predictions) - set(val_files))
        raise RuntimeError(f"Inference file mismatch. missing={missing[:5]}, extra={extra[:5]}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return predictions, elapsed

## 6. baseline / tune_a / tune_b 비교 및 저장

In [13]:
all_rows = []
policy_tables = {}
details = {}

for stage in STAGES:
    info = checkpoint_info[stage]
    print("=" * 80)
    print(f"Evaluating {TARGET_MODEL} / {stage}: {info['path'].name}")

    preds, inference_seconds = infer_checkpoint(info["path"])
    threshold, top_k, comp_map, policy_table = select_policy(preds)

    row = {
        "model": TARGET_MODEL,
        "stage": stage,
        "experiment_id": info["stem"],
        "checkpoint_path": str(info["path"]),
        "competition_metric": COMPETITION_METRIC,
        "competition_mAP": comp_map,
        "selected_confidence": threshold,
        "selected_top_k": top_k,
        "inference_seconds": inference_seconds,
        "latency_ms_per_image": inference_seconds * 1000.0 / len(val_files),
        "validation_images": len(val_files),
        "split_fingerprint": SPLIT_FINGERPRINT,
    }
    all_rows.append(row)
    policy_tables[stage] = policy_table
    details[stage] = {"predictions": preds}
    print(
        f"{stage}: competition mAP={comp_map:.6f}, "
        f"confidence={threshold}, top_k={top_k}, "
        f"inference={inference_seconds/60:.1f} min"
    )

leaderboard = pd.DataFrame(all_rows).sort_values(
    ["competition_mAP", "latency_ms_per_image"],
    ascending=[False, True],
).reset_index(drop=True)
display(leaderboard)

best = leaderboard.iloc[0]
selected_stage = str(best["stage"])
selected_experiment_id = str(best["experiment_id"])
selected_checkpoint_path = str(best["checkpoint_path"])
selected_confidence = float(best["selected_confidence"])
selected_top_k = None if pd.isna(best["selected_top_k"]) else int(best["selected_top_k"])
selected_competition_map = float(best["competition_mAP"])

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
base = f"eval_only_{MODEL_PREFIX}_{SPLIT_FINGERPRINT[:16]}_{stamp}"
leaderboard_path = REPORT_DIR / f"{base}_leaderboard.csv"
policy_path = REPORT_DIR / f"{base}_{selected_stage}_policy_grid.csv"
summary_path = REPORT_DIR / f"{base}_summary.json"

leaderboard.to_csv(leaderboard_path, index=False, encoding="utf-8-sig")
policy_tables[selected_stage].to_csv(policy_path, index=False, encoding="utf-8-sig")

summary = {
    "schema_version": 1,
    "mode": "evaluation_only",
    "training_disabled": True,
    "model_name": TARGET_MODEL,
    "competition_metric": COMPETITION_METRIC,
    "competition_iou_thresholds": COMPETITION_IOU_THRESHOLDS,
    "split_policy": "preserve_exact_upstream",
    "split_fingerprint": SPLIT_FINGERPRINT,
    "train_images": len(train_files),
    "validation_images": len(val_files),
    "independent_test_available": False,
    "selected_stage": selected_stage,
    "selected_experiment_id": selected_experiment_id,
    "selected_checkpoint_path": selected_checkpoint_path,
    "selected_confidence": selected_confidence,
    "selected_top_k": selected_top_k,
    "selected_competition_mAP": selected_competition_map,
    "leaderboard_csv": str(leaderboard_path),
    "policy_grid_csv": str(policy_path),
    "created_at": datetime.now().isoformat(),
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("=" * 80)
print(
    f"SELECTED {TARGET_MODEL}: {selected_stage} / "
    f"mAP@[0.75:0.95]={selected_competition_map:.6f} / "
    f"conf={selected_confidence} / top_k={selected_top_k}"
)
print("Saved leaderboard:", leaderboard_path)
print("Saved summary:", summary_path)

# 세 모델이 각각 다른 Colab에서 병렬로 끝났을 경우,
# 마지막으로 끝난 노트북은 가능하면 최신 3개 summary를 모아 전체 순위도 만든다.
try:
    model_summaries = []
    for model_prefix in ("yolo11s", "yolo11m", "yolo12m"):
        candidates = sorted(
            REPORT_DIR.glob(f"eval_only_{model_prefix}_{SPLIT_FINGERPRINT[:16]}_*_summary.json"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if not candidates:
            raise FileNotFoundError(model_prefix)
        model_summaries.append(json.loads(candidates[0].read_text(encoding="utf-8")))

    cross = pd.DataFrame([{
        "model": s["model_name"],
        "selected_stage": s["selected_stage"],
        "selected_experiment_id": s["selected_experiment_id"],
        "competition_mAP": s["selected_competition_mAP"],
        "selected_confidence": s["selected_confidence"],
        "selected_top_k": s["selected_top_k"],
    } for s in model_summaries]).sort_values("competition_mAP", ascending=False)

    cross_path = REPORT_DIR / f"eval_only_3model_final_{SPLIT_FINGERPRINT[:16]}_{stamp}.csv"
    cross.to_csv(cross_path, index=False, encoding="utf-8-sig")
    print("\n3-model final leaderboard:")
    display(cross)
    print("Saved:", cross_path)
except FileNotFoundError:
    print("다른 모델 평가가 아직 끝나지 않아 3-model 합산표는 나중에 생성됩니다.")

# V4: 자동 런타임 해제는 "정상 평가 + 결과 파일 Drive 저장"이 확인된 경우에만 허용한다.
if not leaderboard_path.is_file():
    raise RuntimeError(f"Leaderboard was not saved: {leaderboard_path}")
if not summary_path.is_file():
    raise RuntimeError(f"Summary was not saved: {summary_path}")

EVALUATION_COMPLETED_OK = True
EVALUATION_SUMMARY_PATH = str(summary_path)
print("Evaluation and Drive report saving completed successfully.")


Evaluating YOLO11m / baseline: yolo11m_baseline_21cc7ae361705449_best.pt


YOLO11m inference:   0%|          | 0/2433 [00:00<?, ?image/s]

baseline: competition mAP=0.973039, confidence=0.001, top_k=6, inference=1.7 min
Evaluating YOLO11m / tune_a: yolo11m_tune_a_4abb3f0ea436d561_best.pt


YOLO11m inference:   0%|          | 0/2433 [00:00<?, ?image/s]

tune_a: competition mAP=0.969520, confidence=0.001, top_k=6, inference=1.7 min
Evaluating YOLO11m / tune_b: yolo11m_tune_b_c35a806678e468d7_best.pt


YOLO11m inference:   0%|          | 0/2433 [00:00<?, ?image/s]

tune_b: competition mAP=0.970833, confidence=0.001, top_k=6, inference=1.7 min


,model,stage,experiment_id,checkpoint_path,competition_metric,competition_mAP,selected_confidence,selected_top_k,inference_seconds,latency_ms_per_image,validation_images,split_fingerprint
0,YOLO11m,baseline,yolo11m_baseline_21cc7ae361705449,/content/drive/MyDrive/baby_kangaroo/week2/공통파...,mAP@[0.75:0.95],0.973039,0.001,6,101.181506,41.587138,2433,cc5d16d3fe042c5297aaad5c8f2fe469ebfecfb1268223...
1,YOLO11m,tune_b,yolo11m_tune_b_c35a806678e468d7,/content/drive/MyDrive/baby_kangaroo/week2/공통파...,mAP@[0.75:0.95],0.970833,0.001,6,101.495574,41.716224,2433,cc5d16d3fe042c5297aaad5c8f2fe469ebfecfb1268223...
2,YOLO11m,tune_a,yolo11m_tune_a_4abb3f0ea436d561,/content/drive/MyDrive/baby_kangaroo/week2/공통파...,mAP@[0.75:0.95],0.969520,0.001,6,101.117228,41.560718,2433,cc5d16d3fe042c5297aaad5c8f2fe469ebfecfb1268223...


SELECTED YOLO11m: baseline / mAP@[0.75:0.95]=0.973039 / conf=0.001 / top_k=6
Saved leaderboard: /content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/reports/eval_only_yolo11m_cc5d16d3fe042c52_20260819_012252_leaderboard.csv
Saved summary: /content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/reports/eval_only_yolo11m_cc5d16d3fe042c52_20260819_012252_summary.json
다른 모델 평가가 아직 끝나지 않아 3-model 합산표는 나중에 생성됩니다.
Evaluation and Drive report saving completed successfully.


## 7. 정상 완료 후 GPU 런타임 자동 해제

위 평가와 Drive 결과 저장이 모두 성공한 경우에만 Colab 런타임을 해제합니다.


In [14]:
# 정상 완료된 경우에만 Colab VM 할당을 반환한다.
AUTO_RELEASE_RUNTIME = True

if AUTO_RELEASE_RUNTIME:
    if not globals().get("EVALUATION_COMPLETED_OK", False):
        raise RuntimeError(
            "Evaluation did not complete successfully; runtime will NOT be released automatically."
        )

    summary_check = Path(globals()["EVALUATION_SUMMARY_PATH"])
    if not summary_check.is_file():
        raise RuntimeError(
            f"Saved summary cannot be verified; runtime will NOT be released: {summary_check}"
        )

    print("All results are saved to Drive.")
    print("Releasing Colab runtime now to stop holding the GPU...")
    sys.stdout.flush()
    time.sleep(2)

    from google.colab import runtime
    runtime.unassign()
else:
    print("AUTO_RELEASE_RUNTIME=False: runtime remains connected.")


All results are saved to Drive.
Releasing Colab runtime now to stop holding the GPU...
